https://github.com/langchain-ai/langchain/tree/master/libs/langchain/langchain/retrievers/document_compressors

https://blog.langchain.dev/improving-document-retrieval-with-contextual-compression/

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

#### **Read & Load Data**

In [2]:
from langchain.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:
loader_harrypotter  = PyPDFLoader("../harrypotter_1.pdf")
documnet_harrypotter = loader_harrypotter.load()

In [4]:
print(len(documnet_harrypotter))

250


In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=100,
    length_function = len,
    is_separator_regex = False,
)

In [6]:
text_harrypotter = text_splitter.split_documents(documnet_harrypotter)
# texts = [doc.page_content for doc in text_harrypotter]
len(text_harrypotter)

1155

In [7]:
text_harrypotter[:5]

[Document(metadata={'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Microsoft Word 8.0', 'creationdate': '2001-02-13T16:47:14+00:00', 'subject': 'Harry Potter', 'author': 'J.K. Rowling', 'moddate': '2005-11-26T18:01:39+02:00', 'title': "Harry Potter, Book 1; The Sorcerer's Stone", 'source': '../harrypotter_1.pdf', 'total_pages': 250, 'page': 1, 'page_label': '2'}, page_content="1\nHarry Potter and the Sorcerer's Stone\nCHAPTER ONE\nTHE BOY WHO LIVED\nMr. and Mrs. Dursley, of number four, Privet Drive, were proud to say\nthat they were perfectly normal, thank you very much. They were the last\npeople you'd expect to be involved in anything strange or mysterious,\nbecause they just didn't hold with such nonsense.\nMr. Dursley was the director of a firm called Grunnings, which made\ndrills. He was a big, beefy man with hardly any neck, although he did"),
 Document(metadata={'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Microsoft Word 8.0', 'creationdate': '2001

#### **Load the Embeddings Model**

In [8]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

#### **Vector DB Setup**

In [9]:
from langchain_community.vectorstores import FAISS

In [10]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [
                f"Document {i+1}:\n\n{d.page_content}\nMetadata: {d.metadata}"
                for i, d in enumerate(docs)
            ]
        )
    )

In [11]:
retriever = FAISS.from_documents(text_harrypotter, embeddings).as_retriever(
    search_kwargs={"k": 10}
)

In [12]:
query = "Who gave Harry his first broomstick?"

docs = retriever.invoke(query)

In [13]:
pretty_print_docs(docs)

Document 1:

"That's a broomstick," he said, throwing it back to Harry with a mixture
of jealousy and spite on his face. "You'll be in for it this time,
Potter, first years aren't allowed them."
Ron couldn't resist it.
"It's not any old broomstick," he said, "it's a Nimbus Two Thousand.
What did you say you've got at home, Malfoy, a Comet Two Sixty?" Ron
grinned at Harry. "Comets look flashy, but they're not in the same
league as the Nimbus."
Metadata: {'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Microsoft Word 8.0', 'creationdate': '2001-02-13T16:47:14+00:00', 'subject': 'Harry Potter', 'author': 'J.K. Rowling', 'moddate': '2005-11-26T18:01:39+02:00', 'title': "Harry Potter, Book 1; The Sorcerer's Stone", 'source': '../harrypotter_1.pdf', 'total_pages': 250, 'page': 131, 'page_label': '132'}
----------------------------------------------------------------------------------------------------
Document 2:

learning to play that night. He bolted his dinner that evening wi

In [14]:
original_context_len = len("\n\n".join([d.page_content for i, d in enumerate(docs)]))
original_context_len

4726

#### **Load LLM**

In [15]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.0
)

#### **Chat Chain**

In [16]:
from langchain.chains import RetrievalQA

In [17]:
chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)

In [18]:
query = "Who gave Harry his first broomstick?"

response = chain.invoke(query)

In [19]:
response

{'query': 'Who gave Harry his first broomstick?',
 'result': 'Harry received his first broomstick, a Nimbus Two Thousand, from Professor McGonagall.'}

In [20]:
print(response['result'])

Harry received his first broomstick, a Nimbus Two Thousand, from Professor McGonagall.


#### **Document Compression**

##### **LLMChainExtractor**

In [21]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

In [22]:
compressor = LLMChainExtractor.from_llm(llm)

In [23]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

In [24]:
query = "Who gave Harry his first broomstick?"
compressed_docs = compression_retriever.invoke(query)

In [25]:
compressed_docs

[Document(metadata={'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Microsoft Word 8.0', 'creationdate': '2001-02-13T16:47:14+00:00', 'subject': 'Harry Potter', 'author': 'J.K. Rowling', 'moddate': '2005-11-26T18:01:39+02:00', 'title': "Harry Potter, Book 1; The Sorcerer's Stone", 'source': '../harrypotter_1.pdf', 'total_pages': 250, 'page': 131, 'page_label': '132'}, page_content='Professor McGonagall'),
 Document(metadata={'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Microsoft Word 8.0', 'creationdate': '2001-02-13T16:47:14+00:00', 'subject': 'Harry Potter', 'author': 'J.K. Rowling', 'moddate': '2005-11-26T18:01:39+02:00', 'title': "Harry Potter, Book 1; The Sorcerer's Stone", 'source': '../harrypotter_1.pdf', 'total_pages': 250, 'page': 132, 'page_label': '133'}, page_content='"Potter\'s been sent a broomstick, Professor," said Malfoy quickly. "Yes, yes, that\'s right," said Professor Flitwick, beaming at Harry. "Professor McGonagall told me all about th

In [26]:
pretty_print_docs(compressed_docs)

Document 1:

Professor McGonagall
Metadata: {'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Microsoft Word 8.0', 'creationdate': '2001-02-13T16:47:14+00:00', 'subject': 'Harry Potter', 'author': 'J.K. Rowling', 'moddate': '2005-11-26T18:01:39+02:00', 'title': "Harry Potter, Book 1; The Sorcerer's Stone", 'source': '../harrypotter_1.pdf', 'total_pages': 250, 'page': 131, 'page_label': '132'}
----------------------------------------------------------------------------------------------------
Document 2:

"Potter's been sent a broomstick, Professor," said Malfoy quickly. "Yes, yes, that's right," said Professor Flitwick, beaming at Harry. "Professor McGonagall told me all about the special circumstances, Potter. And what model is it?" "A Nimbus Two Thousand, sit," said Harry, fighting not to laugh at the look of horror on Malfoy's face. "And it's really thanks to Malfoy here that I've got it," he added.
Metadata: {'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': '

In [27]:
llm_chain_extractor_context_len = len("\n\n".join([d.page_content for i, d in enumerate(compressed_docs)]))
llm_chain_extractor_context_len

427

In [28]:
print("Compressed ratio by LLMChainExtractor:", f"{original_context_len/(llm_chain_extractor_context_len + 1e-5):.2f}x")

Compressed ratio by LLMChainExtractor: 11.07x


In [29]:
chain_extractor = RetrievalQA.from_chain_type(llm=llm, retriever=compression_retriever)

In [30]:
query = "Who gave Harry his first broomstick?"

response = chain_extractor.invoke(query)
response

{'query': 'Who gave Harry his first broomstick?',
 'result': 'Professor McGonagall gave Harry his first broomstick, the Nimbus Two Thousand.'}

In [31]:
query = "Why does Uncle Vernon go to extreme lengths to prevent Harry from reading his letters?"
compressed_docs_1 = compression_retriever.invoke(query)

In [32]:
pretty_print_docs(compressed_docs_1)

Document 1:

"Never told him what was in the letter Dumbledore left fer him? I was there! I saw Dumbledore leave it, Dursley! An' you've kept it from him all these years?"
Metadata: {'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Microsoft Word 8.0', 'creationdate': '2001-02-13T16:47:14+00:00', 'subject': 'Harry Potter', 'author': 'J.K. Rowling', 'moddate': '2005-11-26T18:01:39+02:00', 'title': "Harry Potter, Book 1; The Sorcerer's Stone", 'source': '../harrypotter_1.pdf', 'total_pages': 250, 'page': 38, 'page_label': '39'}
----------------------------------------------------------------------------------------------------
Document 2:

"Who'd be writing to you?" sneered Uncle Vernon, shaking the letter open with one hand and glancing at it. His face went from red to green faster than a set of traffic lights. And it didn't stop there. Within seconds it was the grayish white of old porridge. "P-P-Petunia!" he gasped. Dudley tried to grab the letter to read it, but Uncle Ver

In [33]:
query = "Why does Uncle Vernon go to extreme lengths to prevent Harry from reading his letters?"

response = chain_extractor.invoke(query)
response

{'query': 'Why does Uncle Vernon go to extreme lengths to prevent Harry from reading his letters?',
 'result': "Uncle Vernon goes to extreme lengths to prevent Harry from reading his letters because he is trying to keep Harry from discovering information that he believes is dangerous or undesirable. The letters are linked to Harry's true identity and his connection to the wizarding world, which Uncle Vernon wants to keep hidden from him. He fears the implications of Harry learning about his past and the magical world, so he resorts to drastic measures like nailing up the mail slot and tearing the letters into pieces."}

In [34]:
response['result']

"Uncle Vernon goes to extreme lengths to prevent Harry from reading his letters because he is trying to keep Harry from discovering information that he believes is dangerous or undesirable. The letters are linked to Harry's true identity and his connection to the wizarding world, which Uncle Vernon wants to keep hidden from him. He fears the implications of Harry learning about his past and the magical world, so he resorts to drastic measures like nailing up the mail slot and tearing the letters into pieces."

##### **LLMChainFilter**

In [35]:
from langchain.retrievers.document_compressors import LLMChainFilter

In [36]:
filter = LLMChainFilter.from_llm(llm)

In [37]:
compression_retriever_1 = ContextualCompressionRetriever(
    base_compressor=filter, base_retriever=retriever
)

In [38]:
chain_filter = RetrievalQA.from_chain_type(llm=llm, retriever=compression_retriever_1)

In [39]:
query = "Who gave Harry his first broomstick?"
compressed_docs_chain_filter = compression_retriever_1.invoke(query)

In [40]:
pretty_print_docs(compressed_docs_chain_filter)

Document 1:

Professor McGonagall
Harry had difficulty hiding his glee as he handed the note to Ron to
read.
"A Nimbus Two Thousand!" Ron moaned enviously. "I've never even touched
one."
They left the hall quickly, wanting to unwrap the broomstick in private
before their first class, but halfway across the entrance hall they
found the way upstairs barred by Crabbe and Goyle. Malfoy seized the
package from Harry and felt it.
"That's a broomstick," he said, throwing it back to Harry with a mixture
Metadata: {'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Microsoft Word 8.0', 'creationdate': '2001-02-13T16:47:14+00:00', 'subject': 'Harry Potter', 'author': 'J.K. Rowling', 'moddate': '2005-11-26T18:01:39+02:00', 'title': "Harry Potter, Book 1; The Sorcerer's Stone", 'source': '../harrypotter_1.pdf', 'total_pages': 250, 'page': 131, 'page_label': '132'}
----------------------------------------------------------------------------------------------------
Document 2:

132
"Potter

In [41]:
llm_chain_filter_context_len = len("\n\n".join([d.page_content for i, d in enumerate(compressed_docs_chain_filter)]))
llm_chain_filter_context_len

967

In [42]:
print("Compressed ratio by LLMChainFilter:", f"{original_context_len/(llm_chain_filter_context_len + 1e-5):.2f}x")

Compressed ratio by LLMChainFilter: 4.89x


In [43]:
query = "Who gave Harry his first broomstick?"

response = chain_filter.invoke(query)
response

{'query': 'Who gave Harry his first broomstick?',
 'result': 'Professor McGonagall gave Harry his first broomstick, a Nimbus Two Thousand.'}

##### **EmbeddingsFilter**

In [44]:
from langchain.retrievers.document_compressors import EmbeddingsFilter

In [45]:
embeddings_filter = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.5)

In [46]:
compression_retriever_2 = ContextualCompressionRetriever(
    base_compressor=embeddings_filter, base_retriever=retriever
)

In [47]:
chain_embeddings = RetrievalQA.from_chain_type(llm=llm, retriever=compression_retriever_2)

In [48]:
query = "Who gave Harry his first broomstick?"
compressed_docs_embeddings_filter = compression_retriever_2.invoke(query)

In [49]:
pretty_print_docs(compressed_docs_embeddings_filter)

Document 1:

"That's a broomstick," he said, throwing it back to Harry with a mixture
of jealousy and spite on his face. "You'll be in for it this time,
Potter, first years aren't allowed them."
Ron couldn't resist it.
"It's not any old broomstick," he said, "it's a Nimbus Two Thousand.
What did you say you've got at home, Malfoy, a Comet Two Sixty?" Ron
grinned at Harry. "Comets look flashy, but they're not in the same
league as the Nimbus."
Metadata: {'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Microsoft Word 8.0', 'creationdate': '2001-02-13T16:47:14+00:00', 'subject': 'Harry Potter', 'author': 'J.K. Rowling', 'moddate': '2005-11-26T18:01:39+02:00', 'title': "Harry Potter, Book 1; The Sorcerer's Stone", 'source': '../harrypotter_1.pdf', 'total_pages': 250, 'page': 131, 'page_label': '132'}
----------------------------------------------------------------------------------------------------
Document 2:

learning to play that night. He bolted his dinner that evening wi

In [50]:
embeddings_filter_context_len = len("\n\n".join([d.page_content for i, d in enumerate(compressed_docs_embeddings_filter)]))
embeddings_filter_context_len

4726

In [51]:
print("Compressed ratio by EmbeddingsFilter:", f"{original_context_len/(embeddings_filter_context_len + 1e-5):.2f}x")

Compressed ratio by EmbeddingsFilter: 1.00x


In [52]:
query = "Who gave Harry his first broomstick?"

response = chain_embeddings.invoke(query)
response

{'query': 'Who gave Harry his first broomstick?',
 'result': 'Harry received his first broomstick, a Nimbus Two Thousand, from Professor McGonagall.'}

#### **Compression Retrieval Pipeline**

In [54]:
from langchain.retrievers.document_compressors import DocumentCompressorPipeline
from langchain_community.document_transformers import EmbeddingsRedundantFilter

In [55]:
redundant_filter = EmbeddingsRedundantFilter(embeddings=embeddings)
relevant_filter = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.5)

In [56]:
pipeline_compressor = DocumentCompressorPipeline(
    transformers=[text_splitter, redundant_filter, relevant_filter, filter, compressor]
)

In [57]:
compression_retriever_pipeline = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor, base_retriever=retriever
)

In [58]:
query = "Who gave Harry his first broomstick?"
compressed_docs_pipeline = compression_retriever_pipeline.invoke(query)

In [59]:
pretty_print_docs(compressed_docs_pipeline)

Document 1:

Professor McGonagall
Metadata: {'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Microsoft Word 8.0', 'creationdate': '2001-02-13T16:47:14+00:00', 'subject': 'Harry Potter', 'author': 'J.K. Rowling', 'moddate': '2005-11-26T18:01:39+02:00', 'title': "Harry Potter, Book 1; The Sorcerer's Stone", 'source': '../harrypotter_1.pdf', 'total_pages': 250, 'page': 131, 'page_label': '132'}
----------------------------------------------------------------------------------------------------
Document 2:

"Potter's been sent a broomstick, Professor," said Malfoy quickly.  
"Yes, yes, that's right," said Professor Flitwick, beaming at Harry.  
"Professor McGonagall told me all about the special circumstances, Potter. And what model is it?"  
"A Nimbus Two Thousand, sit," said Harry, fighting not to laugh at the look of horror on Malfoy's face. "And it's really thanks to Malfoy here that I've got it," he added.
Metadata: {'producer': 'Acrobat Distiller 4.0 for Windows', 'creat

In [60]:
pipeline_context_len = len("\n\n".join([d.page_content for i, d in enumerate(compressed_docs_pipeline)]))
pipeline_context_len

433

In [61]:
print("Compressed ratio by EmbeddingsFilter:", f"{original_context_len/(pipeline_context_len + 1e-5):.2f}x")

Compressed ratio by EmbeddingsFilter: 10.91x


In [62]:
chain_pipeline = RetrievalQA.from_chain_type(llm=llm, retriever=compression_retriever_pipeline)

In [63]:
query = "Who gave Harry his first broomstick?"

response = chain_pipeline.invoke(query)
response

{'query': 'Who gave Harry his first broomstick?',
 'result': 'Professor McGonagall gave Harry his first broomstick, the Nimbus Two Thousand.'}